# ASR Assignment 2025-26

This notebook has been provided as a template to get you started on the assignment.  Feel free to use it for your development, or do your development directly in Python.

You can find a full description of the assignment [here](https://www.inf.ed.ac.uk/teaching/courses/asr/coursework-2026.html).

You are provided with two Python modules `observation_model.py` and `wer.py`.  The first was described in [Lab 3](https://github.com/geoph9/asr_labs/blob/main/asr_lab3_4.ipynb).  The second can be used to compute the number of substitution, deletion and insertion errors between ASR output and a reference text.

It can be used as follows:

```python
import wer

my_refence = 'A B C'
my_output = 'A C C D'

wer.compute_alignment_errors(my_reference, my_output)
```

This produces a tuple $(s,d,i)$ giving counts of substitution,
deletion and insertion errors respectively - in this example (1, 0, 1).  The function accepts either two strings, as in the example above, or two lists.  Matching is case sensitive.

## Template code

Assuming that you have already made a function to generate an WFST, `create_wfst()` and a decoder class, `MyViterbiDecoder`, you can perform recognition on all the audio files as follows:


In [1]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import math
import time
import openfst_python as fst
import observation_model
import wer

In [3]:
# Copied from asr_lab2_solutions.ipynb
# Just for you to debug your code

def parse_lexicon(lex_file):
    """
    Parse the lexicon file and return it in dictionary form.
    
    Args:
        lex_file (str): filename of lexicon file with structure '<word> <phone1> <phone2>...'
                        eg. peppers p eh p er z

    Returns:
        lex (dict): dictionary mapping words to list of phones
    """
    
    lex = {}  # create a dictionary for the lexicon entries (this could be a problem with larger lexica)
    with open(lex_file, 'r') as f:
        for line in f:
            line = line.split()  # split at each space
            lex[line[0]] = line[1:]  # first field the word, the rest is the phones
    return lex

def generate_symbol_tables(lexicon, n=3):
    '''
    Return word, phone and state symbol tables based on the supplied lexicon
        
    Args:
        lexicon (dict): lexicon to use, created from the parse_lexicon() function
        n (int): number of states for each phone HMM
        
    Returns:
        word_table (fst.SymbolTable): table of words
        phone_table (fst.SymbolTable): table of phones
        state_table (fst.SymbolTable): table of HMM phone-state IDs
    '''
    
    state_table = fst.SymbolTable()
    phone_table = fst.SymbolTable()
    word_table = fst.SymbolTable()
    
    # add empty <eps> symbol to all tables
    state_table.add_symbol('<eps>')
    phone_table.add_symbol('<eps>')
    word_table.add_symbol('<eps>')
    
    for word, phones  in lexicon.items():
        
        word_table.add_symbol(word)
        
        for p in phones: # for each phone
            
            phone_table.add_symbol(p)
            for i in range(1,n+1): # for each state 1 to n
                state_table.add_symbol('{}_{}'.format(p, i))
            
    return word_table, phone_table, state_table


# call these two functions
lex = parse_lexicon('lexicon.txt')
word_table, phone_table, state_table = generate_symbol_tables(lex)

# add silence states (REQUIRED for Task 2)
for i in range(1, 6):
    if state_table.find(f"sil_{i}") == -1:
        state_table.add_symbol(f"sil_{i}")

def generate_phone_wfst(f, start_state, phone, n, self_loop_prob=0.1):
    """
    Generate a WFST representating an n-state left-to-right phone HMM
    
    Args:
        f (fst.Fst()): an FST object, assumed to exist already
        start_state (int): the index of the first state, assmed to exist already
        phone (str): the phone label 
        n (int): number of states for each phone HMM
        
    Returns:
        the final state of the FST
    """
    
    current_state = start_state
    
    for i in range(1, n+1):
        
        in_label = state_table.find('{}_{}'.format(phone, i))
        forward_prob = 1.0 - self_loop_prob

        sl_weight = fst.Weight('log', -math.log(self_loop_prob))  # weight for self-loop
        # self-loop back to current state
        f.add_arc(current_state, fst.Arc(in_label, 0, sl_weight, current_state))
        
        # transition to next state
        
        # we want to output the phone label on the final state
        # note: if outputting words instead this code should be modified
        if i == n:
            out_label = phone_table.find(phone)
        else:
            out_label = 0   # output empty <eps> label
            
        next_state = f.add_state()
        next_weight = fst.Weight('log', -math.log(forward_prob)) # weight to next state
        f.add_arc(current_state, fst.Arc(in_label, out_label, next_weight, next_state))    
       
        current_state = next_state
        
    return current_state

def generate_silence_wfst(f, start_state):
    """
    Generate a simple optional silence HMM using 5 silence states.
    We do not need to match the exact training topology.
    """
    current_state = start_state

    # use 5 silence states: sil_1 ... sil_5
    for i in range(1, 6):
        in_label = state_table.find(f"sil_{i}")

        next_state = f.add_state()

        # self-loop
        f.add_arc(
            current_state,
            fst.Arc(
                in_label,
                0,
                fst.Weight('log', -math.log(0.3)),
                current_state
            )
        )

        # forward transition
        f.add_arc(
            current_state,
            fst.Arc(
                in_label,
                0,
                fst.Weight('log', -math.log(0.7)),
                next_state
            )
        )

        current_state = next_state

    return current_state

def generate_word_wfst(word, self_loop_prob=0.1):
    """
    Generate a WFST for one word, but output the WORD once at the end.
    """
    f = fst.Fst('log')

    start_state = f.add_state()
    f.set_start(start_state)

    current_state = start_state
    phones = lex[word]

    for phone in phones:
        current_state = generate_phone_wfst(
            f, current_state, phone, 3, self_loop_prob=self_loop_prob)

    # add one epsilon-input arc that outputs the whole word
    final_state = f.add_state()
    f.add_arc(
        current_state,
        fst.Arc(
            0,  # epsilon input
            word_table.find(word),  # output the whole word
            fst.Weight('log', 0.0),  # probability 1
            final_state
        )
    )

    f.set_final(final_state)
    return f

def build_unigram_model(audio_to_reference, smoothing=1.0):
    """
    Estimate unigram probabilities from reference transcriptions.

    smoothing=1.0 gives simple add-one smoothing so no in-vocabulary word
    gets zero probability.
    """
    unigram_counts = {word: 0 for word in lex.keys()}
    total_count = 0

    for ref in audio_to_reference.values():
        words = normalize_text(ref).split()
        for w in words:
            if w in unigram_counts:
                unigram_counts[w] += 1
                total_count += 1

    vocab_size = len(unigram_counts)

    unigram_probs = {}
    for word, count in unigram_counts.items():
        unigram_probs[word] = (count + smoothing) / (total_count + smoothing * vocab_size)

    return unigram_probs


def build_bigram_model(audio_to_reference, smoothing=0.0):
    """
    Optional helper for later experiments.
    This builds actual conditional bigram probabilities P(w_i | w_{i-1}),
    although your current WFST structure does not yet use them properly.
    """
    bigram_counts = {}
    unigram_context_counts = {}

    for ref in audio_to_reference.values():
        words = normalize_text(ref).split()

        for i in range(1, len(words)):
            prev_w = words[i - 1]
            curr_w = words[i]

            if prev_w not in bigram_counts:
                bigram_counts[prev_w] = {}

            bigram_counts[prev_w][curr_w] = bigram_counts[prev_w].get(curr_w, 0) + 1
            unigram_context_counts[prev_w] = unigram_context_counts.get(prev_w, 0) + 1

    bigram_probs = {}
    vocab = list(lex.keys())
    vocab_size = len(vocab)

    for prev_w in bigram_counts:
        bigram_probs[prev_w] = {}
        denom = unigram_context_counts[prev_w] + smoothing * vocab_size

        for curr_w in vocab:
            count = bigram_counts[prev_w].get(curr_w, 0)
            bigram_probs[prev_w][curr_w] = (count + smoothing) / denom

    return bigram_probs


def create_wfst(
    continue_prob=1.0,
    self_loop_prob=0.1,
    use_unigram=False,
    use_silence=False,
    bigram_probs=None,
    unigram_probs=None
):
    f = fst.Fst('log')

    start_state = f.add_state()
    f.set_start(start_state)

    continue_weight = fst.Weight('log', -math.log(continue_prob))

    for word, phones in lex.items():
        current_state = start_state

        for phone in phones:
            current_state = generate_phone_wfst(
                f, current_state, phone, 3, self_loop_prob=self_loop_prob
            )

        word_output_state = f.add_state()

        # Per-word output weight
        if use_unigram and unigram_probs is not None:
            prob = unigram_probs.get(word, 1e-12)
            word_weight = fst.Weight('log', -math.log(prob))
        else:
            word_weight = fst.Weight('log', 0.0)

        f.add_arc(
            current_state,
            fst.Arc(
                0,
                word_table.find(word),
                word_weight,
                word_output_state
            )
        )

        f.set_final(word_output_state)

        # Keep this simple for now.
        # We are not claiming this is a real bigram WFST yet.
        f.add_arc(
            word_output_state,
            fst.Arc(
                0,
                0,
                continue_weight,
                start_state
            )
        )

        if use_silence:
            silence_start = f.add_state()

            f.add_arc(
                word_output_state,
                fst.Arc(
                    0,
                    0,
                    fst.Weight('log', 0.0),
                    silence_start
                )
            )

            silence_end = generate_silence_wfst(f, silence_start)

            f.add_arc(
                silence_end,
                fst.Arc(
                    0,
                    0,
                    continue_weight,
                    start_state
                )
            )
            
    f.set_input_symbols(state_table)
    f.set_output_symbols(word_table)

    return f


In [4]:
print(list(lex.items())[:3])
print("num words:", word_table.num_symbols())
print("num phones:", phone_table.num_symbols())
print("num states:", state_table.num_symbols())

[('a', ['ey']), ('of', ['ah', 'v']), ('peck', ['p', 'eh', 'k'])]
num words: 11
num phones: 18
num states: 57


In [5]:
class MyViterbiDecoder:
    NLL_ZERO = 1e10  # approximate -log(0) with a very large number

    def __init__(self, f, audio_file_name, pruning_threshold=None):
        """Set up the decoder class with an audio file and WFST f"""
        self.om = observation_model.ObservationModel()
        self.f = f
        self.forward_computations = 0
        self.pruning_threshold = pruning_threshold

        if audio_file_name:
            self.om.load_audio(audio_file_name)
        else:
            self.om.load_dummy_audio()

        self.initialise_decoding()

    def initialise_decoding(self):
        """set up the values for V_j(0) (as negative log-likelihoods)
        
        """
        
        self.V = []   # stores likelihood along best path reaching state j
        self.B = []   # stores identity of best previous state reaching state j
        self.W = []   # stores output labels sequence along arc reaching j - this removes need for 
                      # extra code to read the output sequence along the best path
        
        for t in range(self.om.observation_length()+1):
            self.V.append([self.NLL_ZERO]*self.f.num_states())
            self.B.append([-1]*self.f.num_states())
            self.W.append([[] for i in range(self.f.num_states())])  #  multiplying the empty list doesn't make multiple
        
        # The above code means that self.V[t][j] for t = 0, ... T gives the Viterbi cost
        # of state j, time t (in negative log-likelihood form)
        # Initialising the costs to NLL_ZERO effectively means zero probability    
        
        # give the WFST start state a probability of 1.0   (NLL = 0.0)
        self.V[0][self.f.start()] = 0.0
        
        # some WFSTs might have arcs with epsilon on the input (you might have already created 
        # examples of these in earlier labs) these correspond to non-emitting states, 
        # which means that we need to process them without stepping forward in time.  
        # Don't worry too much about this!  
        self.traverse_epsilon_arcs(0)        

    def traverse_epsilon_arcs(self, t):
        """Traverse arcs with <eps> on the input at time t
        
        These correspond to transitions that don't emit an observation
        
        We've implemented this function for you as it's slightly trickier than
        the normal case.  You might like to look at it to see what's going on, but
        don't worry if you can't fully follow it.
        
        """
        
        states_to_traverse = list(self.f.states()) # traverse all states
        while states_to_traverse:
            
            # Set i to the ID of the current state, the first 
            # item in the list (and remove it from the list)
            i = states_to_traverse.pop(0)   
        
            # don't bother traversing states which have zero probability
            if self.V[t][i] == self.NLL_ZERO:
                    continue
        
            for arc in self.f.arcs(i):
                
                if arc.ilabel == 0:     # if <eps> transition
                  
                    j = arc.nextstate   # ID of next state  
                
                    if self.V[t][j] > self.V[t][i] + float(arc.weight):
                        
                        # this means we've found a lower-cost path to
                        # state j at time t.  We might need to add it
                        # back to the processing queue.
                        self.V[t][j] = self.V[t][i] + float(arc.weight)
                        
                        # save backtrace information.  In the case of an epsilon transition, 
                        # we save the identity of the best state at t-1.  This means we may not
                        # be able to fully recover the best path, but to do otherwise would
                        # require a more complicated way of storing backtrace information
                        self.B[t][j] = self.B[t][i] 
                        
                        # and save the output labels encountered - this is a list, because
                        # there could be multiple output labels (in the case of <eps> arcs)
                        if arc.olabel != 0:
                            self.W[t][j] = self.W[t][i] + [arc.olabel]
                        else:
                            self.W[t][j] = self.W[t][i]
                        
                        if j not in states_to_traverse:
                            states_to_traverse.append(j)

    

    def forward_step(self, t):
        
        active_costs = [v for v in self.V[t-1] if v != self.NLL_ZERO]
        if not active_costs:
            return

        best_prev_cost = min(active_costs)
          
        for i in self.f.states():

            if self.V[t-1][i] == self.NLL_ZERO:
                continue

            if self.pruning_threshold is not None:
                dynamic_threshold = self.pruning_threshold

                # prune more aggressively in later frames
                if t > self.om.observation_length() * 0.5:
                    dynamic_threshold *= 0.85

                if self.V[t-1][i] > best_prev_cost + dynamic_threshold:
                    continue

            for arc in self.f.arcs(i):
                if arc.ilabel != 0:
                    j = arc.nextstate
                    tp = float(arc.weight)
                    ep = -self.om.log_observation_probability(
                        self.f.input_symbols().find(arc.ilabel), t
                    )
                    self.forward_computations += 1
                    prob = tp + ep + self.V[t-1][i]

                    if prob < self.V[t][j]:
                        self.V[t][j] = prob
                        self.B[t][j] = i

                        if arc.olabel != 0:
                            self.W[t][j] = [arc.olabel]
                        else:
                            self.W[t][j] = []

    def finalise_decoding(self):
        """Add final-state costs and invalidate non-final endings."""
        T = self.om.observation_length()

        for state in self.f.states():
            if self.V[T][state] == self.NLL_ZERO:
                continue

            final_weight = float(self.f.final(state))
            if final_weight == math.inf:
                # cannot end in a non-final state
                self.V[T][state] = self.NLL_ZERO
            else:
                self.V[T][state] += final_weight

        # optional message if nothing reached the end
        if not any(v < self.NLL_ZERO for v in self.V[T]):
            print("No path got to the end of the observations.")

    def decode(self):
        """Run Viterbi forward pass + epsilon traversals, then apply final weights."""
        self.initialise_decoding()
        t = 1
        while t <= self.om.observation_length():
            self.forward_step(t)
            self.traverse_epsilon_arcs(t)
            t += 1
        self.finalise_decoding()

    def backtrace(self):
        """Recover best state sequence and output label sequence."""
        T = self.om.observation_length()

        final_candidates = [
            s for s in self.f.states()
            if float(self.f.final(s)) != math.inf and self.V[T][s] < self.NLL_ZERO
        ]

        if not final_candidates:
            return ([], "")

        best_final_state = min(final_candidates, key=lambda s: self.V[T][s])

        best_state_sequence = [best_final_state]
        best_out_sequence = []

        t = T
        j = best_final_state

        while t >= 0:
            i = self.B[t][j]
            best_state_sequence.append(i)

            best_out_sequence = self.W[t][j] + best_out_sequence

            if i == -1:
                break

            j = i
            t -= 1

        best_state_sequence.reverse()
        best_out_sequence = ' '.join(
            [self.f.output_symbols().find(label) for label in best_out_sequence if label != 0]
        )

        return (best_state_sequence, best_out_sequence)

In [6]:
def normalize_text(text):
    """
    Lowercase and remove simple punctuation so references and hypotheses
    are compared consistently.
    """
    text = text.lower()
    for ch in [".", ",", "?", "!", "'", '"', ":", ";", "…", "-", "(", ")", "[", "]"]:
        text = text.replace(ch, "")
    text = " ".join(text.split())
    return text

In [7]:
import os

def load_transcripts(transcript_file, recordings_dir):
    audio_to_reference = {}

    with open(transcript_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split()
            filename = parts[0]
            transcription = " ".join(parts[1:])
            transcription = normalize_text(transcription)

            full_path = os.path.join(recordings_dir, filename)
            audio_to_reference[full_path] = transcription

    return audio_to_reference

In [8]:
def evaluate_utterance(f, audio_file, reference, pruning_threshold = None):
    """
    Decode one utterance and return metrics.
    """
    decoder = MyViterbiDecoder(f, audio_file, pruning_threshold=pruning_threshold)

    decode_start = time.time()
    decoder.decode()
    decode_end = time.time()

    backtrace_start = time.time()
    _, hypothesis = decoder.backtrace()
    if hypothesis == "":
        print("Warning: pruning removed all valid final paths for this utterance.")
    backtrace_end = time.time()

    reference = normalize_text(reference)
    hypothesis = normalize_text(hypothesis)

    s, d, i = wer.compute_alignment_errors(reference, hypothesis)

    return {
        "audio_file": audio_file,
        "reference": reference,
        "hypothesis": hypothesis,
        "S": s,
        "D": d,
        "I": i,
        "N": len(reference.split()),
        "decode_time_sec": decode_end - decode_start,
        "backtrace_time_sec": backtrace_end - backtrace_start,
        "time_sec": (decode_end - decode_start) + (backtrace_end - backtrace_start),
        "forward_computations": decoder.forward_computations
    }

In [9]:
def evaluate_dataset(
    audio_to_reference,
    continue_prob=1.0,
    self_loop_prob=0.1,
    use_unigram=False,
    use_silence=False,
    pruning_threshold=None,
    bigram_probs=None,
    unigram_probs=None,
    experiment_name=None
):
    """
    Run decoding on all utterances and aggregate Task 1/2/3/4 metrics.
    """
    f = create_wfst(
        continue_prob=continue_prob,
        self_loop_prob=self_loop_prob,
        use_unigram=use_unigram,
        use_silence=use_silence,
        bigram_probs=bigram_probs,
        unigram_probs=unigram_probs
    )

    num_states = f.num_states()
    num_arcs = sum(1 for s in f.states() for _ in f.arcs(s))

    results = []

    total_S = 0
    total_D = 0
    total_I = 0
    total_N = 0
    total_time = 0.0
    total_decode_time = 0.0
    total_backtrace_time = 0.0
    total_forward = 0
    empty_hyp_count = 0

    for audio_file, reference in audio_to_reference.items():
        result = evaluate_utterance(
            f,
            audio_file,
            reference,
            pruning_threshold=pruning_threshold
        )
        results.append(result)

        total_S += result["S"]
        total_D += result["D"]
        total_I += result["I"]
        total_N += result["N"]
        total_time += result["time_sec"]
        total_decode_time += result["decode_time_sec"]
        total_backtrace_time += result["backtrace_time_sec"]
        total_forward += result["forward_computations"]

        if result["hypothesis"] == "":
            empty_hyp_count += 1

        print("=" * 60)
        print("Audio:", result["audio_file"])
        print("Reference:", result["reference"])
        print("Hypothesis:", result["hypothesis"])
        print("S, D, I:", result["S"], result["D"], result["I"])
        print("Decode time:", result["decode_time_sec"])
        print("Backtrace time:", result["backtrace_time_sec"])
        print("Total time:", result["time_sec"])
        print("Forward computations:", result["forward_computations"])

    wer_value = (total_S + total_D + total_I) / total_N if total_N > 0 else float("inf")

    summary = {
        "experiment_name": experiment_name,
        "num_states": num_states,
        "num_arcs": num_arcs,
        "total_S": total_S,
        "total_D": total_D,
        "total_I": total_I,
        "total_N": total_N,
        "WER": wer_value,
        "total_time_sec": total_time,
        "avg_time_sec": total_time / len(results) if results else 0.0,
        "total_decode_time_sec": total_decode_time,
        "avg_decode_time_sec": total_decode_time / len(results) if results else 0.0,
        "total_backtrace_time_sec": total_backtrace_time,
        "avg_backtrace_time_sec": total_backtrace_time / len(results) if results else 0.0,
        "total_forward_computations": total_forward,
        "avg_forward_computations": total_forward / len(results) if results else 0.0,
        "empty_hypothesis_count": empty_hyp_count,
        "pruning_threshold": pruning_threshold,
        "continue_prob": continue_prob,
        "self_loop_prob": self_loop_prob,
        "use_unigram": use_unigram,
        "use_silence": use_silence,
    }

    return results, summary

In [10]:
import os
import wave
import audioop

src_dir = "recordings"
dst_dir = "recordings_fixed"
os.makedirs(dst_dir, exist_ok=True)

for fname in os.listdir(src_dir):
    if not fname.endswith(".wav"):
        continue

    src_path = os.path.join(src_dir, fname)
    dst_path = os.path.join(dst_dir, fname)

    with wave.open(src_path, "rb") as wf:
        n_channels = wf.getnchannels()
        sampwidth = wf.getsampwidth()
        framerate = wf.getframerate()
        n_frames = wf.getnframes()
        frames = wf.readframes(n_frames)

    # Convert stereo to mono if needed
    if n_channels == 2:
        frames = audioop.tomono(frames, sampwidth, 0.5, 0.5)
        n_channels = 1

    # Convert sample width to 16-bit if needed
    if sampwidth != 2:
        frames = audioop.lin2lin(frames, sampwidth, 2)
        sampwidth = 2

    # Resample to 16000 Hz if needed
    if framerate != 16000:
        frames, _ = audioop.ratecv(frames, sampwidth, n_channels, framerate, 16000, None)
        framerate = 16000

    with wave.open(dst_path, "wb") as wf:
        wf.setnchannels(n_channels)
        wf.setsampwidth(sampwidth)
        wf.setframerate(framerate)
        wf.writeframes(frames)

print("Done converting WAV files.")
print("Converted files:", len([f for f in os.listdir(dst_dir) if f.endswith('.wav')]))

Done converting WAV files.
Converted files: 48


In [11]:
recordings_dir = "recordings_fixed"
transcript_file = "recordings/my_recordings.txt"

audio_to_reference = load_transcripts(transcript_file, recordings_dir)

print("Number of recordings loaded:", len(audio_to_reference))

missing_files = [path for path in audio_to_reference if not os.path.exists(path)]
print("Missing files:", len(missing_files))
if missing_files[:10]:
    print("Examples of missing files:")
    for p in missing_files[:10]:
        print(p)

for k, v in list(audio_to_reference.items())[:5]:
    print(k, "->", v)

Number of recordings loaded: 48
Missing files: 0
recordings_fixed/1.wav -> peter piper picked
recordings_fixed/2.wav -> picked a peck
recordings_fixed/3.wav -> piper picked a
recordings_fixed/4.wav -> a peck of
recordings_fixed/5.wav -> peck of pickled


In [12]:
def check_transcripts_against_lexicon(audio_to_reference, lexicon):
    bad = []

    for audio_file, ref in audio_to_reference.items():
        words = normalize_text(ref).split()
        oov = [w for w in words if w not in lexicon]
        if oov:
            bad.append((audio_file, ref, oov))

    return bad

bad_transcripts = check_transcripts_against_lexicon(audio_to_reference, lex)

print("Number of transcript lines with OOV words:", len(bad_transcripts))
for item in bad_transcripts[:10]:
    print(item)

Number of transcript lines with OOV words: 0


In [13]:
experiments = [
    {
        "name": "Task 1 baseline",
        "continue_prob": 1.0,
        "self_loop_prob": 0.1,
        "use_unigram": False,
        "use_silence": False,
        "pruning_threshold": None,
        "unigram_probs": None
    },
    {
        "name": "Task 2: loop penalty",
        "continue_prob": 0.1,
        "self_loop_prob": 0.1,
        "use_unigram": False,
        "use_silence": False,
        "pruning_threshold": None,
        "unigram_probs": None
    },
    {
        "name": "Task 2: self-loop 0.4",
        "continue_prob": 0.1,
        "self_loop_prob": 0.4,
        "use_unigram": False,
        "use_silence": False,
        "pruning_threshold": None,
        "unigram_probs": None
    },
    {
        "name": "Task 2: + silence",
        "continue_prob": 0.1,
        "self_loop_prob": 0.4,
        "use_unigram": False,
        "use_silence": True,
        "pruning_threshold": None,
        "unigram_probs": None
    },
]

experiment_summaries = []
all_results = []

for exp in experiments:
    print("\n" + "#" * 70)
    print(exp["name"])
    print("#" * 70)

    results, summary = evaluate_dataset(
        audio_to_reference,
        continue_prob=exp["continue_prob"],
        self_loop_prob=exp["self_loop_prob"],
        use_unigram=exp["use_unigram"],
        use_silence=exp["use_silence"],
        pruning_threshold=exp["pruning_threshold"],
        unigram_probs=exp["unigram_probs"],
        experiment_name=exp["name"]
    )

    experiment_summaries.append(summary)

    for r in results:
        row = dict(r)
        row["experiment_name"] = exp["name"]
        all_results.append(row)

    for k, v in summary.items():
        print(f"{k}: {v}")


######################################################################
Task 1 baseline
######################################################################
Audio: recordings_fixed/1.wav
Reference: peter piper picked
Hypothesis: peppers picked wheres picked of peck of the the the the of peck of peppers peppers pickled piper of peck of the the peter piper wheres peppers picked wheres picked the the the the
S, D, I: 0 0 31
Decode time: 1.7428061962127686
Backtrace time: 0.0005471706390380859
Total time: 1.7433533668518066
Forward computations: 71898
Audio: recordings_fixed/2.wav
Reference: picked a peck
Hypothesis: peppers peter picked peck peppers picked wheres picked peter piper wheres the the the the of peck of the the peppers picked wheres picked peppers picked wheres picked the the the the
S, D, I: 1 0 29
Decode time: 1.1877102851867676
Backtrace time: 0.0004420280456542969
Total time: 1.1881523132324219
Forward computations: 52158
Audio: recordings_fixed/3.wav
Reference: piper pi

Audio: recordings_fixed/18.wav
Reference: piper picked pickled peppers
Hypothesis: peppers picked wheres picked of peck of a a a a of peck of the the the the the the a a a a a a a a the the peppers picked wheres picked the the of peck of peter piper wheres peppers peter picked peck the the peppers picked wheres picked the the the the the the
S, D, I: 1 0 54
Decode time: 1.834118127822876
Backtrace time: 0.0005517005920410156
Total time: 1.834669828414917
Forward computations: 80928
Audio: recordings_fixed/19.wav
Reference: peter piper picked peppers
Hypothesis: peppers picked wheres picked the the the the of peck of peter piper wheres a a of peck of peppers picked wheres picked the the the the peter piper wheres of peck of peter piper wheres peppers picked wheres picked
S, D, I: 0 0 36
Decode time: 1.3652994632720947
Backtrace time: 0.0004749298095703125
Total time: 1.365774393081665
Forward computations: 61188
Audio: recordings_fixed/20.wav
Reference: picked peppers peter piper
Hypoth

Audio: recordings_fixed/33.wav
Reference: peter picked a peck of pickled peppers
Hypothesis: peppers peter picked peck peppers picked wheres picked peter piper wheres peter piper wheres peter piper wheres peter piper wheres the the the the of peck of of peck of a a peppers picked wheres picked of peck of peppers picked wheres picked the the the the a a a a a a the the peppers peppers pickled piper a a peppers picked wheres picked peppers picked of pickled the the a a peppers picked of pickled peppers picked wheres picked the the a a peter piper wheres peppers picked wheres picked the the the the
S, D, I: 0 0 89
Decode time: 3.506291151046753
Backtrace time: 0.0009007453918457031
Total time: 3.5071918964385986
Forward computations: 154218
Audio: recordings_fixed/34.wav
Reference: piper picked peck of pickled peppers
Hypothesis: peppers picked wheres picked peppers peppers pickled piper the the the the the the of peck of peppers picked wheres picked a a a a a a peppers picked wheres pick

Audio: recordings_fixed/44.wav
Reference: peter picked a peck of pickled peppers
Hypothesis: the the peppers picked wheres picked peter piper wheres of peck of the the of peck of peppers picked wheres picked the the the the the the a a a a the the peppers peppers pickled piper the the a a a a peppers picked wheres picked the the peppers picked wheres picked of peck of the the a a a a peter piper wheres peppers picked wheres picked the the peppers picked wheres picked peter piper wheres peppers picked wheres picked peppers picked wheres picked peppers picked wheres picked a a a a a a peppers picked wheres picked
S, D, I: 1 0 93
Decode time: 3.9234166145324707
Backtrace time: 0.0009763240814208984
Total time: 3.9243929386138916
Forward computations: 172278
Audio: recordings_fixed/45.wav
Reference: wheres the peck of pickled peppers
Hypothesis: a a peppers picked wheres picked the the peter piper wheres of peck of the the peter piper wheres of peck of a a a a peppers picked wheres picked 

Audio: recordings_fixed/10.wav
Reference: piper picked peppers
Hypothesis: a a peter piper wheres peter piper wheres of peck of peter piper wheres of peck of the the peppers picked of pickled of peck of a a of peck of peter piper wheres the the
S, D, I: 1 0 33
Decode time: 1.5956826210021973
Backtrace time: 0.00048542022705078125
Total time: 1.596168041229248
Forward computations: 68328
Audio: recordings_fixed/11.wav
Reference: peter piper picked a peck
Hypothesis: of peck of peppers picked wheres picked of peck of peppers peppers pickled piper the the of peck of a a peppers picked wheres picked peppers picked wheres picked the the
S, D, I: 3 0 26
Decode time: 1.5307347774505615
Backtrace time: 0.0004646778106689453
Total time: 1.5311994552612305
Forward computations: 66438
Audio: recordings_fixed/12.wav
Reference: piper picked a peck of
Hypothesis: peppers picked wheres picked a a of peck of the the of peck of the the of peck of of peck of a a peppers picked wheres picked peter piper 

...

Audio: recordings_fixed/9.wav
Reference: peter picked peppers
Hypothesis: the the peppers picked wheres picked a a of peck of peppers peter peppers pickled piper peppers a a
S, D, I: 1 0 16
Decode time: 2.4250121116638184
Backtrace time: 0.0005664825439453125
Total time: 2.4255785942077637
Forward computations: 96888
Audio: recordings_fixed/10.wav
Reference: piper picked peppers
Hypothesis: peter piper wheres peter piper wheres of peck of peppers picked wheres picked of peck of a a peter piper wheres a a
S, D, I: 1 0 20
Decode time: 1.5570080280303955
Backtrace time: 0.00045180320739746094
Total time: 1.557459831237793
Forward computations: 68328
Audio: recordings_fixed/11.wav
Reference: peter piper picked a peck
Hypothesis: peppers picked wheres picked of peck of peppers peppers pickled piper the the of peck of a a peppers picked wheres picked a a
S, D, I: 2 0 19
Decode time: 1.5306651592254639
Backtrace time: 0.0004582405090332031
Total time: 1.531123399734497
Forward computations: 6

Audio: recordings_fixed/31.wav
Reference: peter piper picked a peck of pickled peppers
Hypothesis: the the the the of peck of peppers peppers pickled piper the the of peck of the the of peck of the the peppers picked wheres picked of peck of the the peter piper wheres the the
S, D, I: 4 0 29
Decode time: 2.7035889625549316
Backtrace time: 0.0006384849548339844
Total time: 2.7042274475097656
Forward computations: 113058
Audio: recordings_fixed/32.wav
Reference: wheres the peck of pickled peppers peter piper picked
Hypothesis: the the peter piper wheres of peck of a a of peck of peppers picked of pickled peppers picked wheres picked peter piper wheres the the peppers picked wheres picked peppers peppers pickled piper a a a a
S, D, I: 1 0 29
Decode time: 3.190463066101074
Backtrace time: 0.0006444454193115234
Total time: 3.1911075115203857
Forward computations: 129228
Audio: recordings_fixed/33.wav
Reference: peter picked a peck of pickled peppers
Hypothesis: the the the the of peck of a 

Audio: recordings_fixed/1.wav
Reference: peter piper picked
Hypothesis: a a the the the the of peck of peppers peppers pickled piper of peck of a a
S, D, I: 2 0 15
Decode time: 2.3976993560791016
Backtrace time: 0.0005567073822021484
Total time: 2.3982560634613037
Forward computations: 105448
Audio: recordings_fixed/2.wav
Reference: picked a peck
Hypothesis: a a the the of peck of a a
S, D, I: 1 0 6
Decode time: 1.751002311706543
Backtrace time: 0.00048232078552246094
Total time: 1.7514846324920654
Forward computations: 76308
Audio: recordings_fixed/3.wav
Reference: piper picked a
Hypothesis: a a peter piper wheres of peck of the the the the a a a a
S, D, I: 1 0 13
Decode time: 1.7240819931030273
Backtrace time: 0.0004830360412597656
Total time: 1.724565029144287
Forward computations: 76308
Audio: recordings_fixed/4.wav
Reference: a peck of
Hypothesis: a a of peck of the the a a a a
S, D, I: 0 0 8
Decode time: 1.546687126159668
Backtrace time: 0.00045108795166015625
Total time: 1.54713

Audio: recordings_fixed/25.wav
Reference: wheres the peck piper picked
Hypothesis: a a peter piper wheres of peck of a a peppers peppers pickled piper a a a a
S, D, I: 2 0 13
Decode time: 3.2368011474609375
Backtrace time: 0.0006256103515625
Total time: 3.2374267578125
Forward computations: 139858
Audio: recordings_fixed/26.wav
Reference: wheres peter piper
Hypothesis: a a peter piper wheres peppers the wheres pickled peter peppers peppers pickled piper a a
S, D, I: 0 0 13
Decode time: 2.441105842590332
Backtrace time: 0.0005781650543212891
Total time: 2.4416840076446533
Forward computations: 105448
Audio: recordings_fixed/27.wav
Reference: wheres the pickled peck
Hypothesis: a a peter piper wheres of peck of the the peppers picked wheres picked a a a a
S, D, I: 2 0 14
Decode time: 2.8570547103881836
Backtrace time: 0.0005929470062255859
Total time: 2.857647657394409
Forward computations: 126528
Audio: recordings_fixed/28.wav
Reference: wheres the peck of pickled
Hypothesis: a a peter 

Audio: recordings_fixed/46.wav
Reference: peter piper picked pickled peppers
Hypothesis: a a of peck of of peck of a a the the the the the the peppers picked wheres picked peter piper wheres a a
S, D, I: 3 0 20
Decode time: 4.40380334854126
Backtrace time: 0.0007402896881103516
Total time: 4.40454363822937
Forward computations: 190078
Audio: recordings_fixed/47.wav
Reference: peter piper picked a peck
Hypothesis: a a the the the the peppers peppers pickled piper peppers picked wheres picked of peck of a a
S, D, I: 2 0 14
Decode time: 4.717184066772461
Backtrace time: 0.0007536411285400391
Total time: 4.717937707901001
Forward computations: 203408
Audio: recordings_fixed/48.wav
Reference: wheres the peck peter picked
Hypothesis: a a peter piper wheres of peck of a a the the the the peppers peter picked peck peppers peter picked peck a a peppers peppers pickled piper a a peppers peppers pickled piper a a
S, D, I: 0 0 31
Decode time: 5.274071216583252
Backtrace time: 0.0008409023284912109

In [14]:
task3_experiments = [
    {"name": "Task 3: no pruning", "pruning_threshold": None},
    {"name": "Task 3: pruning 50", "pruning_threshold": 50},
    {"name": "Task 3: pruning 100", "pruning_threshold": 100},
    {"name": "Task 3: pruning 200", "pruning_threshold": 200},
    {"name": "Task 3: pruning 500", "pruning_threshold": 500},
]

task3_summaries = []

for exp in task3_experiments:
    print("\n" + "#" * 70)
    print(exp["name"])
    print("#" * 70)

    results, summary = evaluate_dataset(
        audio_to_reference,
        continue_prob=0.1,
        self_loop_prob=0.4,
        use_unigram=False,
        use_silence=True,
        pruning_threshold=exp["pruning_threshold"],
        unigram_probs=None,
        experiment_name=exp["name"]
    )

    task3_summaries.append(summary)

    for k, v in summary.items():
        print(f"{k}: {v}")


######################################################################
Task 3: no pruning
######################################################################
Audio: recordings_fixed/1.wav
Reference: peter piper picked
Hypothesis: a a the the the the of peck of peppers peppers pickled piper of peck of a a
S, D, I: 2 0 15
Decode time: 2.459665298461914
Backtrace time: 0.0005469322204589844
Total time: 2.460212230682373
Forward computations: 105448
Audio: recordings_fixed/2.wav
Reference: picked a peck
Hypothesis: a a the the of peck of a a
S, D, I: 1 0 6
Decode time: 1.7204513549804688
Backtrace time: 0.0004677772521972656
Total time: 1.720919132232666
Forward computations: 76308
Audio: recordings_fixed/3.wav
Reference: piper picked a
Hypothesis: a a peter piper wheres of peck of the the the the a a a a
S, D, I: 1 0 13
Decode time: 1.7126917839050293
Backtrace time: 0.0004730224609375
Total time: 1.7131648063659668
Forward computations: 76308
Audio: recordings_fixed/4.wav
Reference: 

Audio: recordings_fixed/24.wav
Reference: wheres the peck of peppers
Hypothesis: a a the the peter piper wheres the the of peck of a a a a a a
S, D, I: 1 0 13
Decode time: 2.8741228580474854
Backtrace time: 0.0006089210510253906
Total time: 2.8747317790985107
Forward computations: 126528
Audio: recordings_fixed/25.wav
Reference: wheres the peck piper picked
Hypothesis: a a peter piper wheres of peck of a a peppers peppers pickled piper a a a a
S, D, I: 2 0 13
Decode time: 3.3496310710906982
Backtrace time: 0.0006165504455566406
Total time: 3.350247621536255
Forward computations: 139858
Audio: recordings_fixed/26.wav
Reference: wheres peter piper
Hypothesis: a a peter piper wheres peppers the wheres pickled peter peppers peppers pickled piper a a
S, D, I: 0 0 13
Decode time: 2.5372066497802734
Backtrace time: 0.0005393028259277344
Total time: 2.537745952606201
Forward computations: 105448
Audio: recordings_fixed/27.wav
Reference: wheres the pickled peck
Hypothesis: a a peter piper where

Audio: recordings_fixed/45.wav
Reference: wheres the peck of pickled peppers
Hypothesis: a a peter piper wheres of peck of of peck of a a of peck of peter piper wheres a a a a
S, D, I: 3 0 17
Decode time: 4.531806468963623
Backtrace time: 0.0007297992706298828
Total time: 4.532536268234253
Forward computations: 187288
Audio: recordings_fixed/46.wav
Reference: peter piper picked pickled peppers
Hypothesis: a a of peck of of peck of a a the the the the the the peppers picked wheres picked peter piper wheres a a
S, D, I: 3 0 20
Decode time: 4.285018682479858
Backtrace time: 0.0007402896881103516
Total time: 4.285758972167969
Forward computations: 190078
Audio: recordings_fixed/47.wav
Reference: peter piper picked a peck
Hypothesis: a a the the the the peppers peppers pickled piper peppers picked wheres picked of peck of a a
S, D, I: 2 0 14
Decode time: 4.593876123428345
Backtrace time: 0.0007674694061279297
Total time: 4.594643592834473
Forward computations: 203408
Audio: recordings_fixed

No path got to the end of the observations.
Audio: recordings_fixed/17.wav
Reference: peter picked pickled peppers
Hypothesis: 
S, D, I: 0 4 0
Decode time: 1.1253960132598877
Backtrace time: 0.0002994537353515625
Total time: 1.1256954669952393
Forward computations: 46340
No path got to the end of the observations.
Audio: recordings_fixed/18.wav
Reference: piper picked pickled peppers
Hypothesis: 
S, D, I: 0 4 0
Decode time: 1.2074017524719238
Backtrace time: 0.00034356117248535156
Total time: 1.2077453136444092
Forward computations: 50884
Audio: recordings_fixed/19.wav
Reference: peter piper picked peppers
Hypothesis: a a the the the the of peck of peppers peppers pickled piper peter piper wheres a a
S, D, I: 2 0 14
Decode time: 1.2743046283721924
Backtrace time: 0.0005311965942382812
Total time: 1.2748358249664307
Forward computations: 51120
No path got to the end of the observations.
Audio: recordings_fixed/20.wav
Reference: picked peppers peter piper
Hypothesis: 
S, D, I: 0 4 0
Deco

No path got to the end of the observations.
Audio: recordings_fixed/38.wav
Reference: peter piper picked a peck of peppers
Hypothesis: 
S, D, I: 0 7 0
Decode time: 1.663578748703003
Backtrace time: 0.00031638145446777344
Total time: 1.6638951301574707
Forward computations: 66376
No path got to the end of the observations.
Audio: recordings_fixed/39.wav
Reference: peter picked a peck of peppers piper picked a peck of peppers
Hypothesis: 
S, D, I: 0 12 0
Decode time: 2.276033878326416
Backtrace time: 0.00042819976806640625
Total time: 2.2764620780944824
Forward computations: 93216
Audio: recordings_fixed/40.wav
Reference: piper picked a peck of peppers
Hypothesis: a a peter piper wheres the the a a a a
S, D, I: 4 0 5
Decode time: 1.331096887588501
Backtrace time: 0.0005695819854736328
Total time: 1.3316664695739746
Forward computations: 53346
Audio: recordings_fixed/41.wav
Reference: peter piper picked a peck
Hypothesis: a a the the of peck of a a peppers picked wheres picked of peck of 

...

Audio: recordings_fixed/6.wav
Reference: of pickled peppers
Hypothesis: a a of peck of the the peppers picked wheres picked peter piper wheres a a
S, D, I: 1 0 13
Decode time: 1.866734504699707
Backtrace time: 0.00048422813415527344
Total time: 1.8672187328338623
Forward computations: 77096
Audio: recordings_fixed/7.wav
Reference: pickled peppers peter
Hypothesis: a a of peck of peppers peter peppers pickled piper peppers the the a a
S, D, I: 1 0 12
Decode time: 2.559109687805176
Backtrace time: 0.0005645751953125
Total time: 2.5596742630004883
Forward computations: 106418
Audio: recordings_fixed/8.wav
Reference: peppers peter piper
Hypothesis: a a peter piper wheres the the peppers peppers pickled piper a a
S, D, I: 1 0 10
Decode time: 1.8519620895385742
Backtrace time: 0.0007643699645996094
Total time: 1.8527264595031738
Forward computations: 76272
Audio: recordings_fixed/9.wav
Reference: peter picked peppers
Hypothesis: a a the the peppers peter peppers pickled piper peppers a a
S, 

Audio: recordings_fixed/29.wav
Reference: wheres the peppers
Hypothesis: a a peter piper wheres of peck of a a a a
S, D, I: 2 0 9
Decode time: 2.1450908184051514
Backtrace time: 0.0005354881286621094
Total time: 2.1456263065338135
Forward computations: 87136
Audio: recordings_fixed/30.wav
Reference: wheres the peck of peppers peter picked
Hypothesis: a a peter piper wheres of peck of a a peppers peter peppers pickled piper peppers peppers the wheres pickled peter a a a a
S, D, I: 2 0 18
Decode time: 3.617438554763794
Backtrace time: 0.0006632804870605469
Total time: 3.6181018352508545
Forward computations: 156014
Audio: recordings_fixed/31.wav
Reference: peter piper picked a peck of pickled peppers
Hypothesis: a a the the the the of peck of peppers peppers pickled piper the the of peck of the the the the peppers picked wheres picked of peck of the the peter piper wheres a a
S, D, I: 4 0 28
Decode time: 3.7897913455963135
Backtrace time: 0.0007317066192626953
Total time: 3.7905230522155

Audio: recordings_fixed/1.wav
Reference: peter piper picked
Hypothesis: a a the the the the of peck of peppers peppers pickled piper of peck of a a
S, D, I: 2 0 15
Decode time: 2.4837899208068848
Backtrace time: 0.0005750656127929688
Total time: 2.4843649864196777
Forward computations: 105448
Audio: recordings_fixed/2.wav
Reference: picked a peck
Hypothesis: a a the the of peck of a a
S, D, I: 1 0 6
Decode time: 1.8035783767700195
Backtrace time: 0.0004627704620361328
Total time: 1.8040411472320557
Forward computations: 76308
Audio: recordings_fixed/3.wav
Reference: piper picked a
Hypothesis: a a peter piper wheres of peck of the the the the a a a a
S, D, I: 1 0 13
Decode time: 1.8212053775787354
Backtrace time: 0.0004887580871582031
Total time: 1.8216941356658936
Forward computations: 76308
Audio: recordings_fixed/4.wav
Reference: a peck of
Hypothesis: a a of peck of the the a a a a
S, D, I: 0 0 8
Decode time: 1.6021926403045654
Backtrace time: 0.0004630088806152344
Total time: 1.6026

Audio: recordings_fixed/25.wav
Reference: wheres the peck piper picked
Hypothesis: a a peter piper wheres of peck of a a peppers peppers pickled piper a a a a
S, D, I: 2 0 13
Decode time: 3.4747092723846436
Backtrace time: 0.0006213188171386719
Total time: 3.4753305912017822
Forward computations: 139858
Audio: recordings_fixed/26.wav
Reference: wheres peter piper
Hypothesis: a a peter piper wheres peppers the wheres pickled peter peppers peppers pickled piper a a
S, D, I: 0 0 13
Decode time: 2.4598701000213623
Backtrace time: 0.0006325244903564453
Total time: 2.4605026245117188
Forward computations: 105448
Audio: recordings_fixed/27.wav
Reference: wheres the pickled peck
Hypothesis: a a peter piper wheres of peck of the the peppers picked wheres picked a a a a
S, D, I: 2 0 14
Decode time: 3.202761173248291
Backtrace time: 0.0006422996520996094
Total time: 3.2034034729003906
Forward computations: 126528
Audio: recordings_fixed/28.wav
Reference: wheres the peck of pickled
Hypothesis: a a

Audio: recordings_fixed/46.wav
Reference: peter piper picked pickled peppers
Hypothesis: a a of peck of of peck of a a the the the the the the peppers picked wheres picked peter piper wheres a a
S, D, I: 3 0 20
Decode time: 4.438750267028809
Backtrace time: 0.0007486343383789062
Total time: 4.4394989013671875
Forward computations: 190078
Audio: recordings_fixed/47.wav
Reference: peter piper picked a peck
Hypothesis: a a the the the the peppers peppers pickled piper peppers picked wheres picked of peck of a a
S, D, I: 2 0 14
Decode time: 4.884815454483032
Backtrace time: 0.0007443428039550781
Total time: 4.885559797286987
Forward computations: 203408
Audio: recordings_fixed/48.wav
Reference: wheres the peck peter picked
Hypothesis: a a peter piper wheres of peck of a a the the the the peppers peter picked peck peppers peter picked peck a a peppers peppers pickled piper a a peppers peppers pickled piper a a
S, D, I: 0 0 31
Decode time: 5.730734586715698
Backtrace time: 0.0008432865142822

In [15]:
unigram_probs = build_unigram_model(audio_to_reference, smoothing=1.0)

print("Estimated unigram probabilities:")
for word, prob in sorted(unigram_probs.items(), key=lambda x: x[1], reverse=True):
    print(f"{word:10s} -> {prob:.4f}")

Estimated unigram probabilities:
picked     -> 0.1498
peck       -> 0.1377
peppers    -> 0.1215
peter      -> 0.1134
piper      -> 0.1093
a          -> 0.0891
of         -> 0.0891
pickled    -> 0.0810
wheres     -> 0.0567
the        -> 0.0526


In [16]:
uniform_unigram_probs = {word: 1.0 / len(lex) for word in lex.keys()}

task4_experiments = [
    {
        "name": "No word prior",
        "use_unigram": False,
        "unigram_probs": None
    },
    {
        "name": "Uniform word prior",
        "use_unigram": True,
        "unigram_probs": uniform_unigram_probs
    },
    {
        "name": "Estimated unigram prior",
        "use_unigram": True,
        "unigram_probs": unigram_probs
    }
]

task4_summaries = []
task4_results = []

for exp in task4_experiments:
    print("\n" + "#" * 70)
    print(exp["name"])
    print("#" * 70)

    results, summary = evaluate_dataset(
        audio_to_reference,
        continue_prob=0.1,
        self_loop_prob=0.4,
        use_unigram=exp["use_unigram"],
        use_silence=True,
        pruning_threshold=None,
        unigram_probs=exp["unigram_probs"],
        experiment_name=exp["name"]
    )

    task4_summaries.append(summary)

    for r in results:
        row = dict(r)
        row["experiment_name"] = exp["name"]
        task4_results.append(row)

    for k, v in summary.items():
        print(f"{k}: {v}")


######################################################################
No word prior
######################################################################
Audio: recordings_fixed/1.wav
Reference: peter piper picked
Hypothesis: a a the the the the of peck of peppers peppers pickled piper of peck of a a
S, D, I: 2 0 15
Decode time: 2.429277181625366
Backtrace time: 0.0005593299865722656
Total time: 2.4298365116119385
Forward computations: 105448
Audio: recordings_fixed/2.wav
Reference: picked a peck
Hypothesis: a a the the of peck of a a
S, D, I: 1 0 6
Decode time: 1.7328307628631592
Backtrace time: 0.0005056858062744141
Total time: 1.7333364486694336
Forward computations: 76308
Audio: recordings_fixed/3.wav
Reference: piper picked a
Hypothesis: a a peter piper wheres of peck of the the the the a a a a
S, D, I: 1 0 13
Decode time: 1.7243783473968506
Backtrace time: 0.0004932880401611328
Total time: 1.7248716354370117
Forward computations: 76308
Audio: recordings_fixed/4.wav
Reference: 

Audio: recordings_fixed/24.wav
Reference: wheres the peck of peppers
Hypothesis: a a the the peter piper wheres the the of peck of a a a a a a
S, D, I: 1 0 13
Decode time: 3.097372055053711
Backtrace time: 0.0006589889526367188
Total time: 3.0980310440063477
Forward computations: 126528
Audio: recordings_fixed/25.wav
Reference: wheres the peck piper picked
Hypothesis: a a peter piper wheres of peck of a a peppers peppers pickled piper a a a a
S, D, I: 2 0 13
Decode time: 3.2767601013183594
Backtrace time: 0.0006270408630371094
Total time: 3.2773871421813965
Forward computations: 139858
Audio: recordings_fixed/26.wav
Reference: wheres peter piper
Hypothesis: a a peter piper wheres peppers the wheres pickled peter peppers peppers pickled piper a a
S, D, I: 0 0 13
Decode time: 2.4099934101104736
Backtrace time: 0.0005447864532470703
Total time: 2.4105381965637207
Forward computations: 105448
Audio: recordings_fixed/27.wav
Reference: wheres the pickled peck
Hypothesis: a a peter piper wher

Audio: recordings_fixed/45.wav
Reference: wheres the peck of pickled peppers
Hypothesis: a a peter piper wheres of peck of of peck of a a of peck of peter piper wheres a a a a
S, D, I: 3 0 17
Decode time: 4.332897186279297
Backtrace time: 0.0007202625274658203
Total time: 4.333617448806763
Forward computations: 187288
Audio: recordings_fixed/46.wav
Reference: peter piper picked pickled peppers
Hypothesis: a a of peck of of peck of a a the the the the the the peppers picked wheres picked peter piper wheres a a
S, D, I: 3 0 20
Decode time: 4.252820730209351
Backtrace time: 0.0007393360137939453
Total time: 4.2535600662231445
Forward computations: 190078
Audio: recordings_fixed/47.wav
Reference: peter piper picked a peck
Hypothesis: a a the the the the peppers peppers pickled piper peppers picked wheres picked of peck of a a
S, D, I: 2 0 14
Decode time: 4.759558200836182
Backtrace time: 0.0007412433624267578
Total time: 4.760299444198608
Forward computations: 203408
Audio: recordings_fixe

Audio: recordings_fixed/18.wav
Reference: piper picked pickled peppers
Hypothesis: a a a a of peck of a a the the peppers picked wheres picked peppers peter peppers pickled piper peppers a a
S, D, I: 1 0 19
Decode time: 2.687964677810669
Backtrace time: 0.0005741119384765625
Total time: 2.6885387897491455
Forward computations: 118778
Audio: recordings_fixed/19.wav
Reference: peter piper picked peppers
Hypothesis: a a the the the the of peck of peppers peppers pickled piper peter piper wheres a a
S, D, I: 2 0 14
Decode time: 2.177520990371704
Backtrace time: 0.0005109310150146484
Total time: 2.1780319213867188
Forward computations: 89638
Audio: recordings_fixed/20.wav
Reference: picked peppers peter piper
Hypothesis: a a the the peter piper wheres the the the the of peck of peter piper wheres a a
S, D, I: 2 0 15
Decode time: 2.552290678024292
Backtrace time: 0.0005559921264648438
Total time: 2.552846670150757
Forward computations: 110718
Audio: recordings_fixed/21.wav
Reference: wheres 

Audio: recordings_fixed/40.wav
Reference: piper picked a peck of peppers
Hypothesis: a a peter piper wheres the the a a a a
S, D, I: 4 0 5
Decode time: 2.9493210315704346
Backtrace time: 0.0006005764007568359
Total time: 2.9499216079711914
Forward computations: 129318
Audio: recordings_fixed/41.wav
Reference: peter piper picked a peck
Hypothesis: a a the the of peck of a a peppers picked wheres picked of peck of a a peppers picked wheres picked a a
S, D, I: 3 0 19
Decode time: 4.926682472229004
Backtrace time: 0.0008082389831542969
Total time: 4.927490711212158
Forward computations: 203408
Audio: recordings_fixed/42.wav
Reference: peter piper picked a peck
Hypothesis: a a the the peter piper wheres of peck of of peck of the the of peck of a a a a
S, D, I: 2 0 17
Decode time: 4.777697801589966
Backtrace time: 0.0007469654083251953
Total time: 4.778444766998291
Forward computations: 208678
Audio: recordings_fixed/43.wav
Reference: peter piper picked a peck of pickled peppers
Hypothesis: 

Audio: recordings_fixed/13.wav
Reference: picked a peck of pickled
Hypothesis: a a the the of peck of the the of peck of the the of peck of a a
S, D, I: 2 0 14
Decode time: 2.357884168624878
Backtrace time: 0.0005390644073486328
Total time: 2.3584232330322266
Forward computations: 102658
Audio: recordings_fixed/14.wav
Reference: a peck of pickled peppers
Hypothesis: a a the the the the of peck of peppers peter peppers pickled piper peppers a a
S, D, I: 0 0 12
Decode time: 2.894395589828491
Backtrace time: 0.0005710124969482422
Total time: 2.8949666023254395
Forward computations: 118778
Audio: recordings_fixed/15.wav
Reference: peter picked a peck
Hypothesis: a a the the the the a a a a a a
S, D, I: 3 0 8
Decode time: 2.744166612625122
Backtrace time: 0.0005731582641601562
Total time: 2.7447397708892822
Forward computations: 115988
Audio: recordings_fixed/16.wav
Reference: piper picked a peck
Hypothesis: a a peter piper wheres of peck of the the a a
S, D, I: 2 0 8
Decode time: 2.0378961

Audio: recordings_fixed/35.wav
Reference: peter piper picked pickled peppers
Hypothesis: a a the the the the of peck of a a the the a a the the peter piper wheres of peck of peter piper wheres a a
S, D, I: 3 0 23
Decode time: 4.565529823303223
Backtrace time: 0.0007460117340087891
Total time: 4.5662758350372314
Forward computations: 174268
Audio: recordings_fixed/36.wav
Reference: peter picked peppers piper picked peppers
Hypothesis: a a the the the the of peck of a a peppers peter peppers pickled piper peppers peppers peppers pickled piper a a peter piper wheres a a
S, D, I: 3 0 22
Decode time: 6.346116304397583
Backtrace time: 0.0009188652038574219
Total time: 6.34703516960144
Forward computations: 226968
Audio: recordings_fixed/37.wav
Reference: peter piper picked a peck
Hypothesis: a a the the the the of peck of peppers peppers pickled piper a a a a a a
S, D, I: 3 0 14
Decode time: 5.370919942855835
Backtrace time: 0.0009148120880126953
Total time: 5.371834754943848
Forward computa

In [17]:
import pandas as pd

exp_summary_df = pd.DataFrame(experiment_summaries)
task3_summary_df = pd.DataFrame(task3_summaries)
task4_summary_df = pd.DataFrame(task4_summaries)

display_cols = [
    "experiment_name",
    "WER",
    "total_S",
    "total_D",
    "total_I",
    "num_states",
    "num_arcs",
    "avg_decode_time_sec",
    "avg_backtrace_time_sec",
    "avg_time_sec",
    "avg_forward_computations",
    "empty_hypothesis_count"
]

print("Tasks 1-2 summary")
display(exp_summary_df[display_cols].sort_values("WER"))

print("Task 3 summary")
display(task3_summary_df[display_cols].sort_values("WER"))

print("Task 4 summary")
display(task4_summary_df[display_cols].sort_values("WER"))

Tasks 1-2 summary


,experiment_name,WER,total_S,total_D,total_I,num_states,num_arcs,avg_decode_time_sec,avg_backtrace_time_sec,avg_time_sec,avg_forward_computations,empty_hypothesis_count
3,Task 2: + silence,3.637131,99,2,761,176,350,3.266694,0.000630,3.267324,140419.875,0
2,Task 2: self-loop 0.4,4.253165,88,1,919,116,230,2.270320,0.000541,2.270860,95588.625,0
1,Task 2: loop penalty,8.455696,38,0,1966,116,230,2.268882,0.000643,2.269525,95588.625,0
0,Task 1 baseline,11.177215,32,0,2617,116,230,2.175276,0.000656,2.175932,95588.625,0


Task 3 summary


,experiment_name,WER,total_S,total_D,total_I,num_states,num_arcs,avg_decode_time_sec,avg_backtrace_time_sec,avg_time_sec,avg_forward_computations,empty_hypothesis_count
1,Task 3: pruning 50,2.084388,42,147,305,176,350,1.430540,0.000460,1.431000,58061.250000,29
0,Task 3: no pruning,3.637131,99,2,761,176,350,3.299243,0.000645,3.299888,140419.875000,0
2,Task 3: pruning 100,3.637131,99,2,761,176,350,2.429537,0.000649,2.430186,99628.708333,0
3,Task 3: pruning 200,3.637131,99,2,761,176,350,3.156782,0.000646,3.157428,131377.458333,0
4,Task 3: pruning 500,3.637131,99,2,761,176,350,3.391225,0.000630,3.391855,140419.875000,0


Task 4 summary


,experiment_name,WER,total_S,total_D,total_I,num_states,num_arcs,avg_decode_time_sec,avg_backtrace_time_sec,avg_time_sec,avg_forward_computations,empty_hypothesis_count
2,Estimated unigram prior,3.556962,95,2,746,176,350,3.993825,0.000762,3.994588,140419.875,0
1,Uniform word prior,3.569620,95,2,749,176,350,3.250003,0.000636,3.250639,140419.875,0
0,No word prior,3.637131,99,2,761,176,350,3.241525,0.000630,3.242155,140419.875,0


In [18]:
results, summary = evaluate_dataset(
    audio_to_reference,
    continue_prob=0.1,
    self_loop_prob=0.4,
    use_unigram=True,
    use_silence=True,
    pruning_threshold=50,
    unigram_probs=unigram_probs,
    experiment_name="Task 4: pruning 50 + estimated unigram"
)

for k, v in summary.items():
    print(f"{k}: {v}")

No path got to the end of the observations.
Audio: recordings_fixed/1.wav
Reference: peter piper picked
Hypothesis: 
S, D, I: 0 3 0
Decode time: 1.1195390224456787
Backtrace time: 0.000324249267578125
Total time: 1.1198632717132568
Forward computations: 45662
No path got to the end of the observations.
Audio: recordings_fixed/2.wav
Reference: picked a peck
Hypothesis: 
S, D, I: 0 3 0
Decode time: 0.6395905017852783
Backtrace time: 0.0003821849822998047
Total time: 0.6399726867675781
Forward computations: 27124
Audio: recordings_fixed/3.wav
Reference: piper picked a
Hypothesis: a a peter piper wheres of peck of the the the the a a a a
S, D, I: 1 0 13
Decode time: 0.8971960544586182
Backtrace time: 0.00046443939208984375
Total time: 0.897660493850708
Forward computations: 36990
Audio: recordings_fixed/4.wav
Reference: a peck of
Hypothesis: a a of peck of the the a a a a
S, D, I: 0 0 8
Decode time: 0.6541454792022705
Backtrace time: 0.0004634857177734375
Total time: 0.654608964920044
Forw

No path got to the end of the observations.
Audio: recordings_fixed/23.wav
Reference: wheres the peck peter picked
Hypothesis: 
S, D, I: 0 5 0
Decode time: 1.6813600063323975
Backtrace time: 0.0003151893615722656
Total time: 1.6816751956939697
Forward computations: 56952
No path got to the end of the observations.
Audio: recordings_fixed/24.wav
Reference: wheres the peck of peppers
Hypothesis: 
S, D, I: 0 5 0
Decode time: 1.3465871810913086
Backtrace time: 0.000308990478515625
Total time: 1.3468961715698242
Forward computations: 52860
No path got to the end of the observations.
Audio: recordings_fixed/25.wav
Reference: wheres the peck piper picked
Hypothesis: 
S, D, I: 0 5 0
Decode time: 1.3001184463500977
Backtrace time: 0.00039005279541015625
Total time: 1.3005084991455078
Forward computations: 52164
No path got to the end of the observations.
Audio: recordings_fixed/26.wav
Reference: wheres peter piper
Hypothesis: 
S, D, I: 0 3 0
Decode time: 0.8405227661132812
Backtrace time: 0.000

Audio: recordings_fixed/44.wav
Reference: peter picked a peck of pickled peppers
Hypothesis: a a the the the the a a of peck of the the the the peppers picked wheres picked the the peter piper wheres a a
S, D, I: 3 0 19
Decode time: 2.460639238357544
Backtrace time: 0.0009171962738037109
Total time: 2.4615564346313477
Forward computations: 102912
Audio: recordings_fixed/45.wav
Reference: wheres the peck of pickled peppers
Hypothesis: a a peter piper wheres of peck of of peck of a a of peck of peter piper wheres a a a a
S, D, I: 3 0 17
Decode time: 1.7151291370391846
Backtrace time: 0.0007340908050537109
Total time: 1.7158632278442383
Forward computations: 71764
Audio: recordings_fixed/46.wav
Reference: peter piper picked pickled peppers
Hypothesis: a a of peck of of peck of a a the the the the the the peppers picked wheres picked peter piper wheres a a
S, D, I: 3 0 20
Decode time: 1.8912222385406494
Backtrace time: 0.0007646083831787109
Total time: 1.8919868469238281
Forward computatio

In [19]:
results, summary = evaluate_dataset(
    audio_to_reference,
    continue_prob=0.1,
    self_loop_prob=0.4,
    use_unigram=True,
    use_silence=True,
    pruning_threshold=100,
    unigram_probs=unigram_probs,
    experiment_name="Task 4: pruning 100 + estimated unigram"
)

for k, v in summary.items():
    print(f"{k}: {v}")
    
    

Audio: recordings_fixed/1.wav
Reference: peter piper picked
Hypothesis: a a the the the the of peck of peppers peppers pickled piper of peck of a a
S, D, I: 2 0 15
Decode time: 1.8375275135040283
Backtrace time: 0.0005464553833007812
Total time: 1.838073968887329
Forward computations: 74682
Audio: recordings_fixed/2.wav
Reference: picked a peck
Hypothesis: a a the the of peck of a a
S, D, I: 1 0 6
Decode time: 1.4195804595947266
Backtrace time: 0.0004630088806152344
Total time: 1.4200434684753418
Forward computations: 50362
Audio: recordings_fixed/3.wav
Reference: piper picked a
Hypothesis: a a peter piper wheres of peck of the the the the a a a a
S, D, I: 1 0 13
Decode time: 1.4923932552337646
Backtrace time: 0.0005214214324951172
Total time: 1.4929146766662598
Forward computations: 61066
Audio: recordings_fixed/4.wav
Reference: a peck of
Hypothesis: a a of peck of the the a a a a
S, D, I: 0 0 8
Decode time: 1.2857394218444824
Backtrace time: 0.0005059242248535156
Total time: 1.286245

Audio: recordings_fixed/25.wav
Reference: wheres the peck piper picked
Hypothesis: a a peter piper wheres of peck of a a peppers peppers pickled piper a a a a
S, D, I: 2 0 13
Decode time: 2.2952189445495605
Backtrace time: 0.0006594657897949219
Total time: 2.2958784103393555
Forward computations: 95366
Audio: recordings_fixed/26.wav
Reference: wheres peter piper
Hypothesis: a a peter piper wheres peppers the wheres pickled peter peppers peppers pickled piper a a
S, D, I: 0 0 13
Decode time: 1.6454167366027832
Backtrace time: 0.0005626678466796875
Total time: 1.645979404449463
Forward computations: 65830
Audio: recordings_fixed/27.wav
Reference: wheres the pickled peck
Hypothesis: a a peter piper wheres of peck of peppers picked wheres picked a a a a
S, D, I: 3 0 12
Decode time: 2.049598455429077
Backtrace time: 0.0006093978881835938
Total time: 2.0502078533172607
Forward computations: 86204
Audio: recordings_fixed/28.wav
Reference: wheres the peck of pickled
Hypothesis: a a peter piper

Audio: recordings_fixed/46.wav
Reference: peter piper picked pickled peppers
Hypothesis: a a of peck of of peck of a a the the the the the the peppers picked wheres picked peter piper wheres a a
S, D, I: 3 0 20
Decode time: 3.616088390350342
Backtrace time: 0.0007700920104980469
Total time: 3.61685848236084
Forward computations: 135456
Audio: recordings_fixed/47.wav
Reference: peter piper picked a peck
Hypothesis: a a the the the the peppers peppers pickled piper peppers picked wheres picked of peck of a a
S, D, I: 2 0 14
Decode time: 3.020778179168701
Backtrace time: 0.0007967948913574219
Total time: 3.0215749740600586
Forward computations: 125988
Audio: recordings_fixed/48.wav
Reference: wheres the peck peter picked
Hypothesis: a a peter piper wheres of peck of a a the the the the peppers peter picked peck peppers peter picked peck a a peppers peppers pickled piper a a
S, D, I: 0 0 25
Decode time: 3.9913089275360107
Backtrace time: 0.0008671283721923828
Total time: 3.992176055908203


In [20]:
results, summary = evaluate_dataset(
    audio_to_reference,
    continue_prob=0.1,
    self_loop_prob=0.4,
    use_unigram=True,
    use_silence=True,
    pruning_threshold=150,
    unigram_probs=unigram_probs,
    experiment_name="Task 4: pruning 150 + estimated unigram"
)

for k, v in summary.items():
    print(f"{k}: {v}")

Audio: recordings_fixed/1.wav
Reference: peter piper picked
Hypothesis: a a the the the the of peck of peppers peppers pickled piper of peck of a a
S, D, I: 2 0 15
Decode time: 2.2174737453460693
Backtrace time: 0.0006213188171386719
Total time: 2.218095064163208
Forward computations: 88830
Audio: recordings_fixed/2.wav
Reference: picked a peck
Hypothesis: a a the the of peck of a a
S, D, I: 1 0 6
Decode time: 1.5649993419647217
Backtrace time: 0.0004680156707763672
Total time: 1.565467357635498
Forward computations: 62514
Audio: recordings_fixed/3.wav
Reference: piper picked a
Hypothesis: a a peter piper wheres of peck of the the the the a a a a
S, D, I: 1 0 13
Decode time: 1.6607258319854736
Backtrace time: 0.0004851818084716797
Total time: 1.6612110137939453
Forward computations: 69504
Audio: recordings_fixed/4.wav
Reference: a peck of
Hypothesis: a a of peck of the the a a a a
S, D, I: 0 0 8
Decode time: 1.4719462394714355
Backtrace time: 0.0004589557647705078
Total time: 1.4724051

Audio: recordings_fixed/25.wav
Reference: wheres the peck piper picked
Hypothesis: a a peter piper wheres of peck of a a peppers peppers pickled piper a a a a
S, D, I: 2 0 13
Decode time: 2.9942851066589355
Backtrace time: 0.0006499290466308594
Total time: 2.9949350357055664
Forward computations: 119998
Audio: recordings_fixed/26.wav
Reference: wheres peter piper
Hypothesis: a a peter piper wheres peppers the wheres pickled peter peppers peppers pickled piper a a
S, D, I: 0 0 13
Decode time: 2.0113611221313477
Backtrace time: 0.0005729198455810547
Total time: 2.0119340419769287
Forward computations: 83594
Audio: recordings_fixed/27.wav
Reference: wheres the pickled peck
Hypothesis: a a peter piper wheres of peck of peppers picked wheres picked a a a a
S, D, I: 3 0 12
Decode time: 2.5797741413116455
Backtrace time: 0.0006093978881835938
Total time: 2.580383539199829
Forward computations: 104868
Audio: recordings_fixed/28.wav
Reference: wheres the peck of pickled
Hypothesis: a a peter pi

Audio: recordings_fixed/46.wav
Reference: peter piper picked pickled peppers
Hypothesis: a a of peck of of peck of a a the the the the the the peppers picked wheres picked peter piper wheres a a
S, D, I: 3 0 20
Decode time: 4.072170734405518
Backtrace time: 0.0007746219635009766
Total time: 4.0729453563690186
Forward computations: 164794
Audio: recordings_fixed/47.wav
Reference: peter piper picked a peck
Hypothesis: a a the the the the peppers peppers pickled piper peppers picked wheres picked of peck of a a
S, D, I: 2 0 14
Decode time: 3.8600990772247314
Backtrace time: 0.0007832050323486328
Total time: 3.86088228225708
Forward computations: 162580
Audio: recordings_fixed/48.wav
Reference: wheres the peck peter picked
Hypothesis: a a peter piper wheres of peck of a a the the the the peppers peter picked peck peppers peter picked peck a a peppers peppers pickled piper a a
S, D, I: 0 0 25
Decode time: 5.068397283554077
Backtrace time: 0.000896453857421875
Total time: 5.069293737411499
F

In [21]:
results, summary = evaluate_dataset(
    audio_to_reference,
    continue_prob=0.1,
    self_loop_prob=0.4,
    use_unigram=True,
    use_silence=True,
    pruning_threshold=200,
    unigram_probs=unigram_probs,
    experiment_name="Task 4: pruning 200 + estimated unigram"
)

for k, v in summary.items():
    print(f"{k}: {v}")

Audio: recordings_fixed/1.wav
Reference: peter piper picked
Hypothesis: a a the the the the of peck of peppers peppers pickled piper of peck of a a
S, D, I: 2 0 15
Decode time: 2.414669990539551
Backtrace time: 0.0005743503570556641
Total time: 2.4152443408966064
Forward computations: 95968
Audio: recordings_fixed/2.wav
Reference: picked a peck
Hypothesis: a a the the of peck of a a
S, D, I: 1 0 6
Decode time: 1.632629632949829
Backtrace time: 0.0004985332489013672
Total time: 1.6331281661987305
Forward computations: 68802
Audio: recordings_fixed/3.wav
Reference: piper picked a
Hypothesis: a a peter piper wheres of peck of the the the the a a a a
S, D, I: 1 0 13
Decode time: 1.762570858001709
Backtrace time: 0.0005228519439697266
Total time: 1.7630937099456787
Forward computations: 73938
Audio: recordings_fixed/4.wav
Reference: a peck of
Hypothesis: a a of peck of the the a a a a
S, D, I: 0 0 8
Decode time: 1.5469207763671875
Backtrace time: 0.00048232078552246094
Total time: 1.5474030

Audio: recordings_fixed/25.wav
Reference: wheres the peck piper picked
Hypothesis: a a peter piper wheres of peck of a a peppers peppers pickled piper a a a a
S, D, I: 2 0 13
Decode time: 3.183412790298462
Backtrace time: 0.0006427764892578125
Total time: 3.1840555667877197
Forward computations: 131106
Audio: recordings_fixed/26.wav
Reference: wheres peter piper
Hypothesis: a a peter piper wheres peppers the wheres pickled peter peppers peppers pickled piper a a
S, D, I: 0 0 13
Decode time: 2.2803564071655273
Backtrace time: 0.0005652904510498047
Total time: 2.280921697616577
Forward computations: 95102
Audio: recordings_fixed/27.wav
Reference: wheres the pickled peck
Hypothesis: a a peter piper wheres of peck of peppers picked wheres picked a a a a
S, D, I: 3 0 12
Decode time: 2.7659709453582764
Backtrace time: 0.0005939006805419922
Total time: 2.7665648460388184
Forward computations: 115182
Audio: recordings_fixed/28.wav
Reference: wheres the peck of pickled
Hypothesis: a a peter pip

Audio: recordings_fixed/46.wav
Reference: peter piper picked pickled peppers
Hypothesis: a a of peck of of peck of a a the the the the the the peppers picked wheres picked peter piper wheres a a
S, D, I: 3 0 20
Decode time: 4.331728219985962
Backtrace time: 0.0007848739624023438
Total time: 4.332513093948364
Forward computations: 180336
Audio: recordings_fixed/47.wav
Reference: peter piper picked a peck
Hypothesis: a a the the the the peppers peppers pickled piper peppers picked wheres picked of peck of a a
S, D, I: 2 0 14
Decode time: 4.400898694992065
Backtrace time: 0.0008060932159423828
Total time: 4.401704788208008
Forward computations: 184366
Audio: recordings_fixed/48.wav
Reference: wheres the peck peter picked
Hypothesis: a a peter piper wheres of peck of a a the the the the peppers peter picked peck peppers peter picked peck a a peppers peppers pickled piper a a
S, D, I: 0 0 25
Decode time: 5.284684658050537
Backtrace time: 0.0008885860443115234
Total time: 5.285573244094849
F